# Notebook 9: Machine Learning Evaluation and Comparison

## Objective

This notebook conducts the final out-of-sample evaluation of the forecasting models developed in Notebook 08.

The analysis uses the locked January–December 2025 test period to:

1. evaluate the persistence, Ridge, Random Forest, and XGBoost forecasts
2. compare symmetric and asymmetric exchange-rate representations
3. examine performance across food subclasses and months
4. identify where forecasting errors are concentrate
5. compare machine-learning forecasts with time-aligned econometric baselines

No models are tuned or selected using the test results.

In [1]:
# import libraries
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

sns.set_theme(style="whitegrid")

print("Evaluation libraries imported successfully.")

Evaluation libraries imported successfully.


In [4]:
# define input and output locations
processed_data_directory = Path("../data/processed")
ml_table_directory = Path("../reports/tables/machine_learning")
econometric_table_directory = Path(
    "../reports/tables/econometrics"
)

evaluation_table_directory = Path(
    "../reports/tables/model_evaluation"
)
evaluation_figure_directory = Path(
    "../reports/figures/model_evaluation"
)

evaluation_table_directory.mkdir(parents=True, exist_ok=True)
evaluation_figure_directory.mkdir(parents=True, exist_ok=True)


# load the locked outcomes and modelling outputs
ml_data = pd.read_csv(
    processed_data_directory / "ml_model_data.csv",
    parse_dates=["Date"],
)

ml_test_predictions = pd.read_csv(
    ml_table_directory / "ml_test_predictions.csv",
    parse_dates=["Date"],
)

validation_model_results = pd.read_csv(
    ml_table_directory / "validation_model_comparison.csv"
)

test_actuals = ml_data.loc[
    ml_data["Split"] == "Test",
    [
        "Date",
        "ClassDescription",
        "SubclassDescription",
        "Food_Inflation_Pct",
    ],
].copy()

print("Machine-learning data loaded:", len(ml_data))
print("Locked test outcomes loaded:", len(test_actuals))
print("Forecast rows loaded:", len(ml_test_predictions))
print("Validation models loaded:", len(validation_model_results))

Machine-learning data loaded: 4554
Locked test outcomes loaded: 552
Forecast rows loaded: 3864
Validation models loaded: 7


In [3]:
# validate the evaluation sample
identifier_columns = [
    "Date",
    "ClassDescription",
    "SubclassDescription",
]

prediction_identifier_columns = identifier_columns + [
    "Model",
    "Representation",
]

prediction_counts = (
    ml_test_predictions.groupby(
        ["Model", "Representation"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Predictions"})
)

evaluation_checks = pd.DataFrame(
    {
        "Check": [
            "Test sample contains 552 outcomes",
            "Test sample covers 46 food subclasses",
            "Test sample covers 12 months",
            "Test outcomes contain no duplicate keys",
            "Predictions contain seven model variants",
            "Every model variant contains 552 predictions",
            "Predictions contain no duplicate records",
            "Target was absent from prediction file",
        ],
        "Passed": [
            len(test_actuals) == 552,
            test_actuals["SubclassDescription"].nunique() == 46,
            test_actuals["Date"].nunique() == 12,
            not test_actuals.duplicated(identifier_columns).any(),
            len(prediction_counts) == 7,
            prediction_counts["Predictions"].eq(552).all(),
            not ml_test_predictions.duplicated(
                prediction_identifier_columns
            ).any(),
            "Food_Inflation_Pct"
            not in ml_test_predictions.columns,
        ],
    }
)

if not evaluation_checks["Passed"].all():
    failed_checks = evaluation_checks.loc[
        ~evaluation_checks["Passed"],
        "Check",
    ].tolist()
    raise ValueError(f"Evaluation checks failed: {failed_checks}")


# Attach actual outcomes for final evaluation
forecast_evaluation_data = ml_test_predictions.merge(
    test_actuals,
    on=identifier_columns,
    how="left",
    validate="many_to_one",
)

missing_actuals = int(
    forecast_evaluation_data["Food_Inflation_Pct"]
    .isna()
    .sum()
)

if missing_actuals:
    raise ValueError(
        f"{missing_actuals} predictions could not be matched "
        "to test outcomes."
    )

evaluation_sample_summary = pd.DataFrame(
    {
        "Value": [
            len(test_actuals),
            test_actuals["SubclassDescription"].nunique(),
            test_actuals["Date"].nunique(),
            test_actuals["Date"].min(),
            test_actuals["Date"].max(),
            len(prediction_counts),
            len(forecast_evaluation_data),
            missing_actuals,
        ]
    },
    index=[
        "Test outcomes",
        "Food subclasses",
        "Unique months",
        "Start date",
        "End date",
        "Forecast variants",
        "Forecast-evaluation rows",
        "Missing matched outcomes",
    ],
)

display(evaluation_checks)
display(evaluation_sample_summary)
display(prediction_counts)

print(
    "All evaluation setup checks passed:",
    bool(evaluation_checks["Passed"].all()),
)

,Check,Passed
0,Test sample contains 552 outcomes,True
1,Test sample covers 46 food subclasses,True
2,Test sample covers 12 months,True
3,Test outcomes contain no duplicate keys,True
4,Predictions contain seven model variants,True
5,Every model variant contains 552 predictions,True
6,Predictions contain no duplicate records,True
7,Target was absent from prediction file,True


,Value
Test outcomes,552
Food subclasses,46
Unique months,12
Start date,2025-01-01 00:00:00
End date,2025-12-01 00:00:00
Forecast variants,7
Forecast-evaluation rows,3864
Missing matched outcomes,0


,Model,Representation,Predictions
0,Persistence,Lag-1 benchmark,552
1,Random Forest,Asymmetric,552
2,Random Forest,Symmetric,552
3,Ridge,Asymmetric,552
4,Ridge,Symmetric,552
5,XGBoost,Asymmetric,552
6,XGBoost,Symmetric,552


All evaluation setup checks passed: True
